In [14]:
import pandas as pd
import numpy as np

# -------- settings --------
RANDOM_SEED = 42
N_SAMPLE = 30

CONFIG = {
    "medgemma_lay": (
        "/Users/joannehui/Desktop/fyp/padchest/Reports_public/reports_500_medgemma_lay.csv",
        ["medgemma_lay_en"],
    ),
    "mistral_lay": (
        "/Users/joannehui/Desktop/fyp/padchest/Reports_public/mistral_clean_lay.csv",  # Updated path
        ["lay_translation"],
    ),
    "mistral_zh_direct": (
        "/Users/joannehui/Desktop/fyp/padchest/Reports_public/reports_500_mistral_zh.csv",
        ["mistral_zh_cn", "mistral_zh_hk"],  
    ),
    "medgemma_zh_direct": (
        "/Users/joannehui/Desktop/fyp/padchest/Reports_public/output_medgemma_zh.csv",
        ["zh_cn", "zh_hk"],
    ),
    "medgemma_nllb_bi": (
        "/Users/joannehui/Desktop/fyp/padchest/Reports_public/reports_500_medgemma_nllb_bi.csv",
        ["medgemma_lay_en", "nllb_zh_Hans_medgemma", "nllb_zh_Hant_medgemma"],
    ),
    "medgemma_opusmt_bi": (
        "/Users/joannehui/Desktop/fyp/padchest/Reports_public/reports_500_medgemma_opusmt_bi.csv",
        ["medgemma_lay_en", "opusmt_zh_cn_medgemma_v1", "opusmt_zh_cn_medgemma_v2"],
    ),
    "mistral_nllb_bi": (
        "/Users/joannehui/Desktop/fyp/padchest/Reports_public/reports_500_mistral_nllb_zh_clean.csv",
        ["lay_translation", "lay_zh_cn", "lay_zh_tw"],
    ),
    "mistral_opusmt_bi": (
        "/Users/joannehui/Desktop/fyp/padchest/Reports_public/reports_500_translated_mistral_opusmt_zh.csv",
        ["lay_translation", "opus_lay_zh_cn_like", "opus_lay_zh_tw_like"],
    ),
}

ID_COLS = ["StudyID", "ImageID", "label", "sentence_en"]

np.random.seed(RANDOM_SEED)
writer = pd.ExcelWriter("evaluation_template.xlsx", engine="xlsxwriter")

for sheet_name, (csv_file, out_cols) in CONFIG.items():
    print(f"Processing {sheet_name} from {csv_file}")

    try:
        df = pd.read_csv(csv_file)
    except FileNotFoundError:
        print(f"  ERROR: {csv_file} not found—skipping")
        continue

    # Filter: drop rows where sentence_en is blank/NaN/empty
    df = df.dropna(subset=['sentence_en']).query("sentence_en != '' and sentence_en.str.strip() != ''")

    # Column checks
    missing_ids = [c for c in ID_COLS if c not in df.columns]
    if missing_ids:
        print(f"  WARNING: Missing IDs: {missing_ids}")
    
    missing_outs = [c for c in out_cols if c not in df.columns]
    if missing_outs:
        print(f"  WARNING: Missing outputs: {missing_outs}")

    print(f"  → {len(df)} valid rows after filtering blanks")

    # Sample only if enough valid rows
    if len(df) == 0:
        print(f"  ERROR: No valid rows with sentence_en—skipping")
        continue
    elif len(df) <= N_SAMPLE:
        sample_df = df.copy()
    else:
        sample_df = df.sample(N_SAMPLE, random_state=RANDOM_SEED)

    cols_to_keep = [c for c in ID_COLS if c in sample_df.columns] + out_cols
    sheet = sample_df[cols_to_keep].copy()

    # Single rater columns (R1)
    for out in out_cols:
        base = out.replace("/", "_").replace(" ", "_")[:15]
        sheet[f"{base}_Comprehensibility_(1-5)_R1"] = ""
        sheet[f"{base}_WrongFinding_(Yes/No)_R1"] = ""
        sheet[f"{base}_MissingImportantInfo_(Yes/No)_R1"] = ""
        sheet[f"{base}_SafeForPatient_(Yes/No)_R1"] = ""

    sheet.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"  → {len(sheet)} rows with R1 columns")

writer.close()
print("Saved evaluation_template.xlsx")



Processing medgemma_lay from /Users/joannehui/Desktop/fyp/padchest/Reports_public/reports_500_medgemma_lay.csv
  → 417 valid rows after filtering blanks
  → 30 rows with R1 columns
Processing mistral_lay from /Users/joannehui/Desktop/fyp/padchest/Reports_public/mistral_clean_lay.csv
  → 417 valid rows after filtering blanks
  → 30 rows with R1 columns
Processing mistral_zh_direct from /Users/joannehui/Desktop/fyp/padchest/Reports_public/reports_500_mistral_zh.csv
  → 417 valid rows after filtering blanks
  → 30 rows with R1 columns
Processing medgemma_zh_direct from /Users/joannehui/Desktop/fyp/padchest/Reports_public/output_medgemma_zh.csv
  → 417 valid rows after filtering blanks
  → 30 rows with R1 columns
Processing medgemma_nllb_bi from /Users/joannehui/Desktop/fyp/padchest/Reports_public/reports_500_medgemma_nllb_bi.csv
  → 417 valid rows after filtering blanks
  → 30 rows with R1 columns
Processing medgemma_opusmt_bi from /Users/joannehui/Desktop/fyp/padchest/Reports_public/repo

In [6]:
pip install notebook ipykernel pandas openpyxl xlsxwriter


  Using cached xlsxwriter-3.2.9-py3-none-any.whl.metadata (2.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl.metadata (2.5 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 33.3 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 37.5 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 31.7 MB/s  0:00:00eta 0:00:01
Using cached xlsxwriter-3.2.9-py3-none-any.whl (175 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 34.3 MB/s  0:00:00eta 0:00:01
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
pip install textstat


In [ ]:
import pandas as pd
from textstat import textstat

english_targets = {
    "reports_500_medgemma_lay.csv": ["medgemma_lay_en"],
    "reports_500_translated_mistral_lay.csv": ["lay_translation"],
    "reports_500_medgemma_nllb_bi.csv": ["medgemma_lay_en"],
    "reports_500_medgemma_opusmt_bi.csv": ["medgemma_lay_en"],
    "reports_500_translated_mistral_nllb_zh.csv": ["lay_translation"],
    "reports_500_translated_zh_mistral_opus.csv": ["lay_translation"],
}

results = []

for fname, cols in english_targets.items():
    df = pd.read_csv(fname)
    for col in cols:
        # drop NaN or empty
        texts = df[col].fillna("").astype(str)
        fk_scores = []
        for t in texts:
            if t.strip():
                fk_scores.append(textstat.flesch_kincaid_grade(t))
            else:
                fk_scores.append(float("nan"))
        df[f"{col}_FK"] = fk_scores

        desc = df[f"{col}_FK"].describe()
        results.append({
            "file": fname,
            "column": col,
            "count": desc["count"],
            "mean_FK": desc["mean"],
            "std_FK": desc["std"],
            "min_FK": desc["min"],
            "p25_FK": desc["25%"],
            "median_FK": desc["50%"],
            "p75_FK": desc["75%"],
            "max_FK": desc["max"],
        })

summary = pd.DataFrame(results)
summary
